# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**My lane:** Lane 2 — Refresh / Content Opportunity Scoring

**ML Task Type:** **Ranking** (with a binary classification component)

**Why ranking?**
- The output is a **prioritized queue** of pages, ordered by likelihood of being declining
- A content reviewer starts at the top and works down
- The business value is in the **ordering**, not just a yes/no prediction
- Precision@K is the right metric because only the top K pages get reviewed

**Binary classification component:**
- Behind the ranking, I'm predicting a binary label: `is_declining` (True/False)
- The ranking is built from the predicted probability of decline

**Why not just classification?**
- Classification alone doesn't give us the order of priority
- Ranking matches the real decision process: "which page should I look at FIRST?"

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**My target:** `is_declining_label`

**Definition:** A page is "declining" if its trend direction is "down" over a defined time window.

**How I'll define it:**

Using the starter dataset's precomputed `trend_direction` column:
- `trend_direction == "down"` → declining (target = 1)
- `trend_direction != "down"` → not declining (target = 0)

**Why this is a proxy (not the ideal target):**
- `trend_direction` is calculated from the current window, not a future outcome
- A stronger target would be: "page declines over the NEXT 30 days" (future-looking)
- But for this framing, the starter proxy is fine—we'll upgrade to a future-looking target in later weeks

**Future improvement (Week 3+):**

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric:** **Precision@K**

| Metric | What it measures | Why for this problem |
|--------|------------------|----------------------|
| **Precision@50** | Of the top 50 ranked pages, how many are actually declining? | A reviewer checks ~50 pages; we want to maximize true problems found |
| **Average Precision** | The entire ranking quality | Captures whether true positives appear high in the list |
| **ROC-AUC** | Overall model discrimination | Secondary metric for model quality |

**Why Precision@K is the right metric:**
- The real decision is: "reviewer has capacity for K pages per week"
- We want to surface as many true problems as possible in those K slots
- False positives waste reviewer time (cost)
- False negatives mean missed opportunities (cost)

**Target values to beat:**

| Method | Precision@50 |
|--------|--------------|
| Baseline (hand-written rules) | 0.240 |
| Random Forest (starter) | 0.740 |

**My goal:** Achieve Precision@50 > 0.740 on a proper future-looking target.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis:** One row = one content page (`content_hash_id` + `client_hash_id`)

**Why this unit?**
- Each page has its own performance metrics (impressions, clicks, position, etc.)
- Each page has its own content metadata (age, word count, etc.)
- The decision is: "which pages should a reviewer look at?"
- Pages are independent units of analysis

In [8]:
# Load the starter dataset and show the unit of analysis
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

print("=" * 60)
print("📊 UNIT OF ANALYSIS: ONE ROW = ONE CONTENT PAGE")
print("=" * 60)
print(f"\nTotal rows (pages): {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"\nFirst 5 rows (showing key columns):")
print("=" * 60)

# Show key columns to understand the unit
# Using CORRECT column names from your dataset
key_cols = ['content_id', 'client_id', 'impressions_90d', 'clicks_90d', 
            'avg_position', 'ctr', 'trend_direction', 'content_age_days']
df[key_cols].head()

📊 UNIT OF ANALYSIS: ONE ROW = ONE CONTENT PAGE

Total rows (pages): 30,000
Total columns: 44

First 5 rows (showing key columns):


,content_id,client_id,impressions_90d,clicks_90d,avg_position,ctr,trend_direction,content_age_days
0,content_304f48230142,client_f369cb89fc,3803,29,10.6,0.76,down,187
1,content_a1fb4e703a9e,client_4e07408562,15320,7,20.3,0.05,down,445
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,0.09,down,141
3,content_331d6c4de07b,client_19581e27de,11751,58,6.2,0.49,stable,463
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,44.0,0.13,down,263


In [9]:
# Show the label distribution
print("=" * 60)
print(" LABEL DISTRIBUTION")
print("=" * 60)
print(df['trend_direction'].value_counts())
print(f"\nDeclining percentage: {df[df['trend_direction'] == 'down'].shape[0] / len(df) * 100:.1f}%")
print(f"Non-declining percentage: {df[df['trend_direction'] != 'down'].shape[0] / len(df) * 100:.1f}%")

 LABEL DISTRIBUTION
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining percentage: 54.2%
Non-declining percentage: 45.8%


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**The current baseline (hand-written rule):**

```python
baseline_refresh_score =
  0.40 * visibility_score
+ 0.30 * freshness_risk_score
+ 0.25 * position_opportunity_score
+ 0.05 * depth_gap_score

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.